# AI 드론 데이터 코딩 실습 노트 (실습용)
### Ⅲ. 드론 시뮬레이션

이 노트는 교재의 예제를 **직접 채워 넣으며** 익히는 실습용입니다.

1. 셀을 **위에서부터 차례대로** 실행합니다. (**Shift + Enter**)
2. 코드 셀에서 `#` 설명을 읽고 **비어 있는 곳**(`=` 뒤, 설명 아래 빈 줄, 빈 괄호)을 채웁니다.
3. 셀 아래 **실행 결과**와 같게 나오면 성공입니다. 막히면 교재의 같은 번호 예제를 봅니다.
4. `해 보기`는 정답이 하나가 아닙니다. 바꿔 보고, 결과가 왜 달라졌는지 말해 봅시다.

> 빈칸을 채우기 전에 실행하면 빨간 오류가 나옵니다. 고장이 아니라 '아직 덜 썼다'는 뜻입니다.

| 소단원 | 예제 |
|---|---|
| 3-1 드론 비행 시뮬레이션 | 3-1, 3-2, 3-3, 3-4 |
| 3-2 비행 결과 비교와 시각화 | 3-5, 3-6, 3-7 |
| 3-3 지도와 공간정보 활용 | 3-8, 3-9, 3-10, 3-11, 3-12 |
| 3-4 배송 경로와 거점 후보 검토 | 3-13, 3-14, 3-15 |

## 0. 준비하기

아래 셀은 **실습 준비용**입니다. 내용을 이해하지 않아도 괜찮습니다. 실행만 하면 다음 일을 대신 해 줍니다.

- 가상 드론 `drone_sim.py`를 인터넷에서 내려받습니다. 내려받지 못했을 때만 **파일 선택 창**이 뜹니다. 그때는 선생님이 나누어 준 파일을 골라 올립니다.
- 그래프에 한글이 깨지지 않게 해 주는 `font_setup.py`를 만들어 둡니다.

> 코랩은 연결이 끊기면 파일을 잊습니다. 다음 시간에 다시 열었다면 이 셀부터 다시 실행합니다.

In [ ]:
#@title ▶ 준비 셀 (실행만 하세요)
import os, importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# 1) 한글 글꼴 설정 모듈(font_setup.py)을 만들어 둔다 (교재 부록 2와 같은 내용)
FONT_SETUP = '''"""font_setup.py - 그래프에 한글이 깨지지 않도록 글꼴을 정해 주는 모듈
그래프를 그리는 예제에서 import font_setup 한 줄만 적으면 된다.
코랩에는 한글 글꼴이 없으므로, 처음 한 번 나눔고딕을 설치한다.
"""
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# 1) 코랩이면 나눔고딕을 설치해서 등록한다
if os.path.exists("/content") and not os.path.exists(NANUM):
    os.system("apt-get -qq install fonts-nanum > /dev/null")
if os.path.exists(NANUM):
    fm.fontManager.addfont(NANUM)

# 2) 운영체제마다 들어 있는 한글 글꼴이 다르므로, 있는 것을 찾아 쓴다
candidates = ["NanumGothic", "Malgun Gothic", "AppleGothic", "Noto Sans CJK KR"]
installed = {f.name for f in fm.fontManager.ttflist}
for name in candidates:
    if name in installed:
        plt.rcParams["font.family"] = name
        break
else:
    print("한글 글꼴을 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")

plt.rcParams["axes.unicode_minus"] = False   # 음수 기호가 깨지지 않도록
'''
if not os.path.exists("font_setup.py"):
    with open("font_setup.py", "w", encoding="utf-8") as f:
        f.write(FONT_SETUP)

# 2) 가상 드론(drone_sim.py)을 내려받는다. 받지 못하면 파일 선택 창을 띄운다
import urllib.request
URL = "https://raw.githubusercontent.com/ralralra/dronebook/main/data/drone_sim.py"
try:
    urllib.request.urlretrieve(URL, "drone_sim.py")
except Exception:
    print("drone_sim.py 를 내려받지 못했습니다.")
if not os.path.exists("drone_sim.py") and IN_COLAB:
    from google.colab import files
    print("선생님이 나누어 준 drone_sim.py 를 골라 올려 주세요.")
    files.upload()

for name in ["drone_sim.py", "font_setup.py"]:
    print(("✓ 준비됨  " if os.path.exists(name) else "✗ 없음    ") + name)


# 3-1 드론 비행 시뮬레이션

**학습 목표**

- 시뮬레이션의 뜻과 쓰임을 설명할 수 있습니다.
- 비행 시나리오를 설계하고 프로그램으로 구현할 수 있습니다.
- 조건을 바꿔 반복 실행하고 결과를 표로 모을 수 있습니다.

<sub>관련 성취기준 [드코1 03-01-01] / [드코1 03-01-02]</sub>

> **생각 열기**  
> '속도를 바꾸면 배터리를 얼마나 더 쓸까?' 이 질문에 답하려고 실제 드론을 열 번 날린다면 배터리를 여러 번 충전해야 합니다. 코드론에듀는 한 번 충전에 60분이 걸립니다. 컴퓨터 안에서라면 몇 초면 끝납니다.

<sub>교재에서 읽기: ■ 1. 시뮬레이션이란</sub>

## ■ 2. 비행 시나리오 설계하기

프로그램을 바로 만들기 전에 무엇을, 어떤 순서로, 어떤 조건에서 할지 정합니다. 이것이 비행 시나리오입니다. 다음과 같이 표로 정리하면 좋습니다.

| 항목 | 내용 |
|---|---|
| 임무 | 창고에서 출발해 두 곳에 물품을 배송하고 돌아옵니다 |
| 경로 | 앞으로 200 cm → 오른쪽 돌아 150 cm → 오른쪽 돌아 200 cm → 오른쪽 돌아 150 cm |
| 조건 | 속도 0.5 m/s, 바람 없음 |
| 확인할 것 | 총 비행 시간, 배터리 소모량, 출발지로 정확히 돌아오는가 |

<sub>표 3-2. 비행 시나리오 설계 예시</sub>

**예제 3-1. 배송 시나리오 시뮬레이션**

In [ ]:
# 예제 3-1. 배송 시나리오 시뮬레이션
from drone_sim import *

route = [200, 150, 200, 150]        # 각 구간의 거리(cm)

drone = Drone()
drone.pair()
drone.takeoff()
# route에서 거리를 하나씩 꺼내며 반복

    # 속도 0.5 m/s로 그 거리만큼 앞으로
    
    # 오른쪽으로 90도
    
print("최종 위치    :", drone.get_pos_x(), drone.get_pos_y())
drone.land()

print("총 비행 시간 :", drone.sim_log[-1]["시간"], "초")    # 가상 전용
print("남은 배터리  :", drone.get_battery(), "%")
drone.close()

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 200cm
[가상] 오른쪽으로 90도 회전
[가상] 앞으로 150cm
[가상] 오른쪽으로 90도 회전
[가상] 앞으로 200cm
[가상] 오른쪽으로 90도 회전
[가상] 앞으로 150cm
[가상] 오른쪽으로 90도 회전
최종 위치    : 0.0 -0.0
[가상] 착륙
총 비행 시간 : 23.0 초
남은 배터리  : 94 %
[가상] 연결을 닫았습니다.
```

최종 위치가 0이므로 출발지로 정확히 돌아왔습니다. 비행 시간 23초에는 이륙(2초)과 이륙 뒤 안정화(1초), 착륙(2초)도 들어 있습니다. 남은 18초 가운데 14초는 7 m를 0.5 m/s로 날아간 시간이고, 4초는 네 번 회전한 시간입니다.

## ■ 3. 조건을 바꿔 반복 실행하기

시뮬레이션의 진짜 힘은 같은 비행을 조건만 바꿔 되풀이할 때 나옵니다. 바깥 for문이 조건을 바꾸고, 안쪽 for문이 같은 경로를 날게 합니다. 매번 Drone()을 새로 만들어 배터리가 100%인 새 드론으로 출발합니다.

> **안전 수칙 · 한 번에 한 가지만 바꿉니다.**  
> 속도와 바람을 동시에 바꾸면, 결과가 달라졌을 때 무엇 때문인지 알 수 없습니다. 속도를 비교할 때는 바람을 고정하고, 바람을 비교할 때는 속도를 고정합니다. 과학 실험에서 변인을 통제하는 것과 같은 원리입니다.

**예제 3-2. 속도를 바꿔 가며 실험하기**

In [ ]:
# 예제 3-2. 속도를 바꿔 가며 실험하기
from drone_sim import *

route = [200, 150, 200, 150]
results = []

# 속도 0.3, 0.5, 0.8을 차례로 실험

    drone = Drone(verbose=False)        # 가상 전용: 비행 안내문 끄기
    drone.pair()
    drone.takeoff()
    for distance in route:
        # 이번 속도(speed)로 그 거리만큼 앞으로
        
        drone.turn_right(90)
    x, y = drone.get_pos_x(), drone.get_pos_y()
    drone.land()
    log = drone.sim_log[-1]             # 가상 전용: 마지막 기록
    results.append({"속도": speed,
                    "비행시간": log["시간"],
                    "배터리소모": round(100 - log["배터리"], 1),
                    "도착위치": (x, y)})
    drone.close()

for r in results:
    print(r)

실행 결과

```
{'속도': 0.3, '비행시간': 32.33, '배터리소모': 7.7, '도착위치': (0.0, -0.0)}
{'속도': 0.5, '비행시간': 23.0, '배터리소모': 5.8, '도착위치': (0.0, -0.0)}
{'속도': 0.8, '비행시간': 17.75, '배터리소모': 4.8, '도착위치': (0.0, -0.0)}
```

x, y = … 처럼 쉼표로 이어 쓰면 값 두 개를 한 줄에 담을 수 있습니다. 결과는 2-4의 예제 2-17과 같은 방향입니다. 빠르게 날수록 비행 시간이 짧아지고, 떠 있는 시간이 줄어 배터리도 덜 썼습니다. 이는 가상 드론에 '떠 있는 동안 계속 배터리를 쓴다'는 규칙이 들어 있기 때문입니다. 시뮬레이션 결과를 해석할 때는 그 속에 어떤 규칙이 들어 있는지 알아야 합니다.

## ■ 4. 바람의 영향 확인하기

이번에는 속도를 고정하고 바람만 바꿔 봅시다. 가상 드론은 바람이 불면 배터리를 더 쓰고 오른쪽으로 밀리도록 만들어져 있습니다.

**예제 3-3. 바람을 바꿔 가며 실험하기**

In [ ]:
# 예제 3-3. 바람을 바꿔 가며 실험하기
from drone_sim import *

route = [200, 150, 200, 150]

# 바람 0, 2, 4, 6을 차례로 실험

    drone = Drone(verbose=False)
    # 가상 드론의 풍속을 wind로 맞추기
    
    drone.pair()
    drone.takeoff()
    for distance in route:
        drone.move_forward(distance, speed=0.5)
        drone.turn_right(90)
    x, y = drone.get_pos_x(), drone.get_pos_y()
    drone.land()
    away =        # 출발지와의 거리
    # 100에서 마지막 배터리를 뺀 값(소수 첫째 자리)
    used =
    print({"바람": wind, "배터리소모": used, "도착위치": (x, y), "출발지와의거리": away})
    drone.close()

실행 결과

```
{'바람': 0, '배터리소모': 5.8, '도착위치': (0.0, -0.0), '출발지와의거리': 0.0}
{'바람': 2, '배터리소모': 6.7, '도착위치': (0.0, -61.96), '출발지와의거리': 62.0}
{'바람': 4, '배터리소모': 7.7, '도착위치': (0.0, -123.92), '출발지와의거리': 123.9}
{'바람': 6, '배터리소모': 8.6, '도착위치': (0.0, -185.88), '출발지와의거리': 185.9}
```

바람이 세질수록 배터리를 더 쓰고, 출발지에서 점점 멀어집니다. 풍속이 2배, 3배가 되자 밀린 거리도 정확히 2배, 3배가 되었습니다. 가상 드론의 바람 규칙이 '풍속에 비례해 밀린다'이기 때문입니다. 실제 바람은 이렇게 고르지 않고, 돌풍처럼 갑자기 세지기도 합니다.

## ■ 5. 결과를 표로 모으기

여러 번 실행한 결과는 데이터프레임으로 모으면 비교하기 쉽고, 다음 소단원에서 그래프로 나타내기에도 좋습니다. 이번에는 for문 안에 for문을 하나 더 넣어 속도 3가지와 바람 3가지를 모두 조합합니다.

**예제 3-4. 실험 결과를 표로 정리하고 저장하기**

In [ ]:
# 예제 3-4. 실험 결과를 표로 정리하고 저장하기
from drone_sim import *
import pandas as pd

route = [200, 150, 200, 150]
results = []

# 속도 세 가지를 차례로

    # 바람 세 가지를 차례로 (반복문 안의 반복문)
    
        drone = Drone(verbose=False)
        drone.sim_set_wind(wind)
        drone.pair()
        drone.takeoff()
        for distance in route:
            drone.move_forward(distance, speed=speed)
            drone.turn_right(90)
        x, y = drone.get_pos_x(), drone.get_pos_y()
        drone.land()
        log = drone.sim_log[-1]
        results.append({"속도": speed, "바람": wind,
                        "비행시간": log["시간"],
                        "배터리소모": round(100 - log["배터리"], 1),
                        "이탈거리": round((x ** 2 + y ** 2) ** 0.5, 1)})
        drone.close()

# results로 데이터프레임 만들기
df =
# sim_result.csv로 저장

print(df)

실행 결과

```
    속도  바람   비행시간  배터리소모   이탈거리
0  0.3   0  32.33    7.7    0.0
1  0.3   3  32.33    9.6  148.4
2  0.3   6  32.33   11.6  296.9
3  0.5   0  23.00    5.8    0.0
4  0.5   3  23.00    7.2   92.9
5  0.5   6  23.00    8.6  185.9
6  0.8   0  17.75    4.8    0.0
7  0.8   3  17.75    5.8   61.7
8  0.8   6  17.75    6.9  123.5
```

속도 3가지 × 바람 3가지 = 9번의 비행 결과가 한 표에 모였습니다. 눈여겨볼 줄이 있습니다. 같은 바람 6 m/s에서도 0.3 m/s로 날면 297 cm, 0.8 m/s로 날면 124 cm 밀렸습니다. 느리게 날수록 바람을 맞는 시간이 길어지기 때문입니다. 이 sim\_result.csv 파일을 다음 소단원에서 그래프로 나타냅니다.

> **한 걸음 더 · 같은 코드를 여러 번 쓰고 있다면**  
> 예제 3-2\~3-4는 '이륙 → 경로 비행 → 위치 읽기 → 착륙'이라는 같은 묶음을 세 번 썼습니다. 'AI 드론 지능형 제어'에서 배울 def를 쓰면 이 묶음에 이름을 붙여 한 줄로 부를 수 있습니다. 지금은 '같은 묶음이 되풀이된다'는 것을 알아차리는 것으로 충분합니다.

### ====== 해 보기 ======

**우리 모둠의 시나리오 만들기**

1. 표 3-2를 참고해 모둠만의 비행 시나리오를 설계합니다. 임무, 경로, 조건, 확인할 것을 적습니다.
2. 경로를 리스트로 만들어 프로그램으로 구현합니다.
3. 바꿔 볼 조건을 한 가지만 정하고, 값을 3\~4개 골라 반복 실행합니다.
4. 결과를 표로 정리하고, 예상과 달랐던 점을 찾아 적습니다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 시뮬레이션은 현실의 일을 컴퓨터 안에서 흉내 내어 미리 결과를 알아보는 것입니다.
- 프로그램을 만들기 전에 임무·경로·조건·확인할 것을 담은 시나리오를 설계합니다.
- 조건을 바꿔 비교할 때는 한 번에 한 가지만 바꿉니다.
- 시뮬레이션 결과는 그 속에 들어 있는 규칙의 범위 안에서만 의미가 있습니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 3-2 비행 결과 비교와 시각화

**학습 목표**

- 비교 목적에 맞는 그래프 종류를 고를 수 있습니다.
- 제목·축 이름·단위를 갖춘 그래프를 그릴 수 있습니다.
- 그래프를 읽고 차이가 나타난 원인을 설명할 수 있습니다.

<sub>관련 성취기준 [드코1 03-02-01] / [드코1 03-02-02]</sub>

> **생각 열기**  
> 앞 소단원에서 아홉 줄짜리 표를 얻었습니다. 표만 보고도 경향을 읽을 수 있을까요? 숫자가 백 개, 천 개가 되면 어떻게 될까요?

<sub>교재에서 읽기: ■ 1. 왜 그래프로 나타내는가 · ■ 2. 그래프 그리기 준비</sub>

## ■ 3. 막대그래프로 비교하기

**예제 3-5. 속도별 비행 시간 비교**

In [ ]:
# 예제 3-5. 속도별 비행 시간 비교
import pandas as pd
import matplotlib.pyplot as plt
import font_setup                         # 한글 글꼴 설정

df = pd.read_csv("sim_result.csv")
# 바람이 0인 줄만 고르기
calm =

plt.figure(figsize=(5, 3))
# 막대그래프: 가로 속도(글자로), 세로 비행시간

# 제목

# 가로축 이름

# 세로축 이름

plt.tight_layout()
plt.savefig("graph_speed.png", dpi=150)
print("그래프를 graph_speed.png 로 저장했습니다.")
print(calm[["속도", "비행시간"]])

실행 결과

```
그래프를 graph_speed.png 로 저장했습니다.
    속도   비행시간
0  0.3  32.33
3  0.5  23.00
6  0.8  17.75
```

<sub>(그림 3-1. 예제 3-5가 만든 막대그래프 — 교재 참고)</sub>

> **짚고 가기 · 속도를 글자로 바꾼 까닭**  
> calm["속도"].astype(str)로 속도를 글자로 바꿨습니다. 숫자 그대로 두면 0.3, 0.5, 0.8이 수직선 위의 위치로 그려져 막대 사이 간격이 제각각이 됩니다. 글자로 바꾸면 세 항목이 나란히 같은 간격으로 놓입니다. 막대그래프는 '항목끼리 비교'가 목적이므로 이쪽이 알맞습니다.

## ■ 4. 꺾은선그래프로 흐름 보기

비행하는 동안 배터리가 어떻게 줄어드는지 보려면 꺾은선그래프가 알맞습니다.

**예제 3-6. 시간에 따른 배터리 변화**

In [ ]:
# 예제 3-6. 시간에 따른 배터리 변화
from drone_sim import *
import pandas as pd
import matplotlib.pyplot as plt
import font_setup

drone = Drone(verbose=False)
drone.pair()
drone.takeoff()
for distance in [200, 150, 200, 150]:
    drone.move_forward(distance, speed=0.5)
    drone.turn_right(90)
drone.land()
drone.sim_save_log()
drone.close()

df = pd.read_csv("flight_log.csv")
plt.figure(figsize=(5, 3))
# 선그래프: 가로 시간, 세로 배터리 (marker="o")

# 제목

# 가로축 이름

# 세로축 이름

plt.grid(True, linewidth=0.5)
plt.tight_layout()
plt.savefig("graph_battery.png", dpi=150)
print("저장 완료. 마지막 배터리:", df["배터리"].iloc[-1], "%")

실행 결과

```
저장 완료. 마지막 배터리: 94.2 %
```

<sub>(그림 3-2. 비행이 진행될수록 배터리가 줄어든다 — 교재 참고)</sub>

선의 기울기가 가파른 구간은 배터리를 빠르게 쓴 구간입니다. 기울기를 읽는 것만으로도 어느 동작이 전력을 많이 쓰는지 짐작할 수 있습니다. df["배터리"].iloc[-1]은 '배터리 열의 마지막 값'입니다. 1-4에서 리스트의 마지막을 [-1]로 꺼낸 것과 같은 생각입니다.

## ■ 5. 여러 조건을 한 그래프에서 비교하기

서로 다른 조건의 결과를 같은 그래프에 겹쳐 그리면 차이가 분명히 드러납니다.

**예제 3-7. 바람에 따른 배터리 소모 비교**

In [ ]:
# 예제 3-7. 바람에 따른 배터리 소모 비교
import pandas as pd
import matplotlib.pyplot as plt
import font_setup

df = pd.read_csv("sim_result.csv")
plt.figure(figsize=(5.2, 3.2))
for speed, mark in [(0.3, "o"), (0.5, "s"), (0.8, "^")]:
    # 속도가 speed인 줄만 고르기
    same =
    # 선그래프: 가로 바람, 세로 배터리소모, 범례 이름(label) 붙이기
    
plt.title("바람에 따른 배터리 소모량")
plt.xlabel("풍속 (m/s)")
plt.ylabel("배터리 소모 (%)")
# 범례 표시

plt.grid(True, linewidth=0.5)
plt.tight_layout()
plt.savefig("graph_wind.png", dpi=150)
print(df.pivot(index="바람", columns="속도", values="배터리소모"))

실행 결과

```
속도   0.3  0.5  0.8
바람
0    7.7  5.8  4.8
3    9.6  7.2  5.8
6   11.6  8.6  6.9
```

<sub>(그림 3-3. 세 조건의 결과를 겹쳐 그린 그래프 — 교재 참고)</sub>

for speed, mark in [(0.3, "o"), …]는 튜플 리스트에서 속도와 점 모양을 한 쌍씩 꺼냅니다. pivot()은 긴 표를 바람을 줄로, 속도를 칸으로 하는 표로 다시 짜 줍니다.

그래프에서 세 선은 모두 오른쪽 위로 올라갑니다. 바람이 세질수록 속도와 상관없이 배터리를 더 씁니다. 그런데 세 선의 높이가 다릅니다. 가장 느린 0.3 m/s가 늘 가장 위에 있습니다. 즉 이 시뮬레이션에서는 바람과 속도가 둘 다 배터리 소모에 영향을 줍니다.

> **한 걸음 더 · 좋은 그래프의 조건**  
> - 제목이 있어 무엇을 나타낸 그래프인지 알 수 있습니다.
> - 축 이름과 단위가 적혀 있습니다. (속도 (m/s))
> - 선이 여러 개면 범례가 있습니다.
> - 색만으로 구분하지 않습니다. 예제 3-7처럼 점 모양(marker)을 함께 다르게 하면 흑백으로 인쇄해도 구분할 수 있습니다.

## ■ 6. 그래프를 읽고 설명하기

그래프를 그리는 것으로 끝이 아닙니다. 무엇이 보이는지, 왜 그런지를 설명할 수 있어야 합니다. 다음 세 단계로 설명하면 좋습니다.

1. 관찰: 무엇이 보이는가? (속도 0.5 m/s에서 바람이 0에서 6으로 갈 때 소모량이 5.8%에서 8.6%로 늘었습니다.)
2. 해석: 그것이 무슨 뜻인가? (바람이 세질수록 더 많은 전력을 씁니다.)
3. 근거와 한계: 왜 그런가? 믿을 만한가? (바람에 밀리지 않으려면 더 큰 힘이 필요합니다. 다만 이는 가상 드론의 규칙에 따른 결과입니다.)

### ====== 해 보기 ======

**그래프 설명문 쓰기**

예제 3-7이 만든 그래프를 보고 위의 세 단계에 따라 설명문을 써 봅시다. 그리고 모둠원과 바꿔 읽으며 '근거와 한계' 부분이 빠지지 않았는지 서로 확인합니다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 표는 정확한 값을, 그래프는 전체 경향과 차이를 보여 줍니다.
- 비교에는 막대그래프, 흐름에는 꺾은선그래프, 관계에는 산점도가 알맞습니다.
- 그래프에는 제목, 축 이름과 단위, 범례를 갖춥니다. 코랩에서는 한글 글꼴을 먼저 설정합니다.
- 그래프는 관찰 → 해석 → 근거와 한계의 순서로 설명합니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 3-3 지도와 공간정보 활용

**학습 목표**

- 공간정보의 뜻과 종류를 설명할 수 있습니다.
- 좌표를 이용해 거리와 경로를 계산할 수 있습니다.
- 비행 경로와 제한 구역을 지도 위에 나타내고 해석할 수 있습니다.

<sub>관련 성취기준 [드코1 03-03-01] / [드코1 03-03-02]</sub>

> **생각 열기**  
> 지도 앱은 목적지까지의 길을 알려 줍니다. 그런데 드론에게 필요한 지도는 조금 다릅니다. 도로가 아니라 하늘길이 필요하고, 어디를 날면 안 되는지도 알아야 합니다.

<sub>교재에서 읽기: ■ 1. 공간정보란</sub>

## ■ 2. 좌표로 거리 계산하기

지도 위의 두 지점을 좌표로 나타내면 거리와 방향을 계산할 수 있습니다. 여기서는 이해를 돕기 위해 가로·세로가 km인 평면 좌표를 씁니다. 2-1에서 배운 math.dist()를 다시 씁니다.

**예제 3-8. 거점에서 배송지까지의 거리**

In [ ]:
# 예제 3-8. 거점에서 배송지까지의 거리
import math

points = {
    "거점A": (1.2, 2.0),
    "배송지1": (3.6, 4.2),
    "배송지2": (6.6, 4.0),
    "배송지3": (7.8, 1.8),
}
# 거점A의 좌표 꺼내기
base =
for name, xy in points.items():
    if name == "거점A":
        continue                  # 거점 자신은 건너뛴다
    # 거점과 배송지 사이의 거리: math.dist
    d =
    print(f"거점A → {name} : {round(d, 2)} km")

실행 결과

```
거점A → 배송지1 : 3.26 km
거점A → 배송지2 : 5.76 km
거점A → 배송지3 : 6.6 km
```

딕셔너리의 items()는 키와 값을 한 쌍씩 꺼내 줍니다. continue는 '이번 반복은 여기서 그만하고 다음 반복으로'라는 뜻입니다. 1-4의 break가 반복을 아예 끝내는 것과 다릅니다.

## ■ 3. 경로의 총 거리 구하기

여러 지점을 차례로 들르는 경로의 총 거리는 이웃한 두 지점 사이의 거리를 모두 더해 구합니다.

**예제 3-9. 배송 경로의 총 거리**

In [ ]:
# 예제 3-9. 배송 경로의 총 거리
import math

points = {"거점A": (1.2, 2.0), "배송지1": (3.6, 4.2),
          "배송지2": (6.6, 4.0), "배송지3": (7.8, 1.8)}
route = ["거점A", "배송지1", "배송지2", "배송지3", "거점A"]

total = 0
for i in range(len(route) - 1):
    # 이번 지점과 다음 지점 사이의 거리
    leg =
    # 총 거리에 더하기
    total =
    print(f"{route[i]} → {route[i+1]} : {round(leg, 2)} km")
print("-" * 30)
print("총 비행 거리:", round(total, 2), "km")
print("속도 10 m/s 로 날면", round(total * 1000 / 10 / 60, 1), "분 걸린다")

실행 결과

```
거점A → 배송지1 : 3.26 km
배송지1 → 배송지2 : 3.01 km
배송지2 → 배송지3 : 2.51 km
배송지3 → 거점A : 6.6 km
------------------------------
총 비행 거리: 15.37 km
속도 10 m/s 로 날면 25.6 분 걸린다
```

경로에 지점이 5개 있으면 구간은 4개입니다. 그래서 range(len(route) - 1)로 한 번 덜 반복합니다. 25.6분은 코드론에듀의 비행 시간(7\~8분)보다 훨씬 깁니다. 이 경로는 더 큰 배송용 드론을 생각하고 계산한 것입니다.

## ■ 4. 비행 제한 구역 확인하기

경로가 제한 구역을 지나가는지 확인해야 합니다. 제한 구역을 중심과 반지름을 가진 원으로 단순하게 나타내면, 어떤 지점이 그 안에 있는지 거리로 판단할 수 있습니다. 중심까지의 거리가 반지름보다 작으면 안입니다.

**예제 3-10. 제한 구역 안에 있는지 확인하기**

In [ ]:
# 예제 3-10. 제한 구역 안에 있는지 확인하기
import math

center, radius = (5.0, 6.0), 1.25          # 제한 구역의 중심 좌표와 반지름(km)
points = {"거점A": (1.2, 2.0), "배송지1": (3.6, 4.2),
          "배송지2": (6.6, 4.0), "배송지3": (7.8, 1.8)}

for name, xy in points.items():
    # 경계까지 남은 거리 = 중심까지의 거리 − 반지름
    gap =
    # gap이 0 이하이면 "제한 구역 안", 아니면 "비행 가능"
    state =
    print(f"{name:6s} {state:8s} (경계까지 {round(gap, 2)} km)")

실행 결과

```
거점A    비행 가능    (경계까지 4.27 km)
배송지1   비행 가능    (경계까지 1.03 km)
배송지2   비행 가능    (경계까지 1.31 km)
배송지3   비행 가능    (경계까지 3.8 km)
```

"제한 구역 안" if gap &lt;= 0 else "비행 가능"은 if문을 한 줄로 줄인 모양입니다. 조건이 참이면 앞의 값을, 거짓이면 뒤의 값을 고릅니다.

모든 지점이 제한 구역 밖에 있습니다. 다만 지점이 밖에 있다고 경로 전체가 안전한 것은 아닙니다. 두 지점을 잇는 직선이 제한 구역을 가로지를 수도 있기 때문입니다.

## ■ 5. 좌표를 그림으로 나타내기

숫자만으로는 경로의 모양을 알기 어렵습니다. 좌표를 그림으로 나타내면 경로와 제한 구역의 관계가 한눈에 보입니다.

**예제 3-11. 비행 경로 그림 그리기**

In [ ]:
# 예제 3-11. 비행 경로 그림 그리기
import matplotlib.pyplot as plt
import font_setup

points = {"거점A": (1.2, 2.0), "배송지1": (3.6, 4.2),
          "배송지2": (6.6, 4.0), "배송지3": (7.8, 1.8)}
route = ["거점A", "배송지1", "배송지2", "배송지3"]

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.add_patch(plt.Circle((5.0, 6.0), 1.25, color="#c0392b", alpha=0.15))   # 비행 제한 구역
ax.text(5.0, 6.0, "비행 제한", ha="center", color="#c0392b")

# 경로 순서대로 x 좌표 모으기
xs =
# 경로 순서대로 y 좌표 모으기
ys =
# 경로를 선으로 잇기 (marker="o")

for name, (x, y) in points.items():
    ax.text(x, y - 0.45, name, ha="center")

ax.set_title("드론 배송 경로")
ax.set_xlabel("동서 방향 (km)")
ax.set_ylabel("남북 방향 (km)")
ax.set_xlim(0, 9); ax.set_ylim(0.5, 8)
ax.set_aspect("equal")
ax.grid(True, linewidth=0.5)
plt.tight_layout()
plt.savefig("map_route.png", dpi=150)
print("지도를 map_route.png 로 저장했습니다.")

실행 결과

```
지도를 map_route.png 로 저장했습니다.
```

<sub>(그림 3-4. 예제 3-11이 만든 비행 경로 그림 — 교재 참고)</sub>

[points[p][0] for p in route]는 '경로의 지점마다 x 좌표만 꺼내 리스트로'라는 뜻입니다. ax.set\_aspect("equal")은 가로 1 km와 세로 1 km를 같은 길이로 그리게 해 원이 찌그러지지 않게 합니다.

그림을 보면 배송지1과 배송지2를 잇는 구간이 제한 구역 바로 아래를 지납니다. 숫자만 보았을 때는 '모두 밖'이었지만, 그림으로 보니 이 구간이 경계에 꽤 가깝습니다. 시각화는 숫자가 놓친 것을 찾아 줍니다.

## ■ 6. 실제 지도 위에 나타내기: plotly

이번에는 진짜 지도 위에 그려 봅시다. plotly는 위도·경도를 넣으면 인터넷 지도 위에 표시해 주는 라이브러리로, 코랩에 설치되어 있습니다. 한꺼번에 완성하지 않고, 지도를 만드는 함수에 인자를 하나씩 더해 가며 지도가 어떻게 달라지는지 확인합니다. 단계마다 셀을 실행하고, 앞 단계의 지도와 견주어 보세요.

### ① 지도를 만들어 변수에 담고 띄우기

px.scatter\_map()은 위도(lat)와 경도(lon)를 받아 그 자리에 점을 찍은 지도를 만드는 함수입니다. 만든 지도를 fig라는 변수에 담아 두고, fig.show()로 화면에 띄웁니다. 점을 찍을 곳은 청주국제공항입니다.

In [ ]:
# ① 지도를 만들어 변수에 담고 띄우기
import plotly.express as px

# 위도 36.7166, 경도 127.4991에 점을 찍은 지도 만들기 (map_style="carto-positron")
fig =
# 지도 띄우기

지도 한가운데에 파란 점이 하나 보입니다. lat과 lon의 값을 [ ]로 감싼 까닭은 점을 여러 개 찍을 수 있도록 리스트로 받기 때문입니다. map\_style은 배경 지도의 종류를 고르는 인자입니다. import 줄은 한 번만 실행하면 되므로 ②부터는 적지 않습니다.

### ② zoom으로 확대하기

①의 지도는 너무 멀리서 본 모습입니다. 함수의 괄호 안에 zoom 인자를 더합니다. 숫자가 클수록 가까이에서 본 지도가 됩니다.

In [ ]:
# ② zoom으로 확대하기
# ①의 함수에 zoom 인자를 더해 12로 확대하기
fig = px.scatter_map(lat=[36.7166], lon=[127.4991], map_style="carto-positron",
                     )
fig.show()

같은 함수에 인자 하나를 더했을 뿐인데 공항 둘레의 길과 건물이 보일 만큼 가까워졌습니다. zoom을 8, 10, 14로 바꿔 가며 실행해 보세요. 인자가 길어지면 이 예제처럼 쉼표 뒤에서 줄을 바꿔 써도 됩니다.

### ③ 점의 색과 크기 바꾸기

점이 작아 눈에 잘 띄지 않습니다. 이미 만든 지도 fig를 고칠 때는 fig.update\_traces()를 씁니다.

In [ ]:
# ③ 점의 색과 크기 바꾸기
fig = px.scatter_map(lat=[36.7166], lon=[127.4991], map_style="carto-positron",
                     zoom=12)
# 점의 크기를 20, 색을 "red"로 바꾸기

fig.show()

marker\_size는 점의 크기, marker\_color는 점의 색입니다. 색은 "blue", "green", "orange"처럼 영어 이름으로 적습니다. 두 값을 바꿔 가며 실행해 보세요.

> **짚고 가기 · 지도를 만드는 함수와 고치는 함수**  
> px.scatter\_map()은 지도를 새로 만들어 돌려주는 함수이고, fig.update\_traces()는 이미 만든 지도 fig를 고치는 함수입니다. 1-5에서 drone.takeoff()가 drone에게 내리는 명령이었던 것처럼, fig.update\_traces()와 fig.show()는 fig에게 내리는 명령입니다.

### ④ 딕셔너리에 여러 장소 담기

장소가 늘어나면 위도와 경도를 매번 숫자로 적기 어렵습니다. 1-4에서 배운 딕셔너리에 '장소 이름: (위도, 경도)'로 담아 두면, 이름만으로 좌표를 꺼낼 수 있습니다. 좌표는 항공정보간행물(AIP)에 실린 각 공항의 기준점을 소수 넷째 자리까지 옮긴 값입니다.

In [ ]:
# ④ 딕셔너리에 여러 장소 담기
places = {
    "청주국제공항": (36.7166, 127.4991),
    "김포국제공항": (37.5569, 126.7975),
    "제주국제공항": (33.5122, 126.4928),
    "김해국제공항": (35.1806, 128.9381),
}

# 이름으로 제주국제공항의 위도와 경도 꺼내기
lat, lon =
fig = px.scatter_map(lat=[lat], lon=[lon], map_style="carto-positron", zoom=12)
fig.update_traces(marker_size=20, marker_color="red")
fig.show()

places["제주국제공항"]은 (33.5122, 126.4928)이라는 튜플을 돌려주고, 이것을 lat과 lon 두 변수에 나누어 담았습니다. 대괄호 안의 이름만 "김포국제공항"으로 바꾸면 지도가 통째로 그곳으로 옮겨 갑니다. 숫자를 직접 고칠 때보다 틀릴 일이 훨씬 적습니다.

### ⑤ 여러 점을 한꺼번에 올리기

이번에는 네 공항을 지도 한 장에 모두 올립니다. for 문으로 딕셔너리의 장소를 하나씩 꺼내 이름, 위도, 경도를 각각 리스트에 모은 다음, 리스트를 통째로 함수에 넘깁니다. ④에서 만든 places를 그대로 씁니다.

In [ ]:
# ⑤ 여러 점을 한꺼번에 올리기
names, lats, lons = [], [], []
for name in places:                      # 딕셔너리에서 장소를 하나씩 꺼낸다
    names.append(name)
    # 위도를 lats에 더하기

    # 경도를 lons에 더하기


fig = px.scatter_map(lat=lats, lon=lons, hover_name=names,
                     map_style="carto-positron", zoom=5)
fig.update_traces(marker_size=12, marker_color="red")
fig.show()

점이 네 개 찍혔습니다. zoom을 5로 줄인 것은 네 곳이 모두 보이게 하기 위해서입니다. hover\_name에 이름 리스트를 넘겼으므로 점 위에 마우스를 올리면 공항 이름이 나옵니다. 딕셔너리에 장소를 하나 더 적고 다시 실행해 보세요. for 문과 그 아래는 한 글자도 고치지 않아도 점이 다섯 개가 됩니다.

### ⑥ 표로 된 자료를 지도에 올리기

공공데이터처럼 위도와 경도가 열로 들어 있는 표는 데이터프레임 그대로 넘길 수 있습니다. 함수의 첫 인자로 데이터프레임을 주고, lat과 lon에는 값 대신 열 이름을 적습니다.

In [ ]:
# ⑥ 표로 된 자료를 지도에 올리기
import pandas as pd

df = pd.DataFrame({
    "공항": ["청주국제공항", "김포국제공항", "제주국제공항", "김해국제공항"],
    "위도": [36.7166, 37.5569, 33.5122, 35.1806],
    "경도": [127.4991, 126.7975, 126.4928, 128.9381],
})

# lat, lon, hover_name에 알맞은 열 이름 적기
fig = px.scatter_map(df, lat=, lon=, hover_name=,
                     map_style="carto-positron", zoom=5)
fig.update_traces(marker_size=12, marker_color="red")
fig.show()

⑤와 똑같은 지도가 나옵니다. 달라진 것은 for 문이 사라졌다는 점입니다. 표를 통째로 넘기면 plotly가 행을 하나씩 읽어 점을 찍어 줍니다. 행이 네 개든 사천 개든 코드는 같습니다. 2-2에서 배운 pd.read\_csv()로 읽은 표도 같은 방법으로 올릴 수 있습니다.

| 단계 | 더한 것 | 지도에서 달라진 것 |
|---|---|---|
| ① | px.scatter\_map(lat, lon, map\_style) | 지도와 점 하나가 나타납니다 |
| ② | zoom | 지도가 가까워집니다 |
| ③ | fig.update\_traces(marker\_size, marker\_color) | 점의 크기와 색이 바뀝니다 |
| ④ | 딕셔너리와 이름으로 꺼내기 | 이름만 바꿔 다른 곳으로 옮겨 갑니다 |
| ⑤ | for 문으로 만든 리스트, hover\_name | 점이 여러 개가 되고 이름이 붙습니다 |
| ⑥ | 데이터프레임과 열 이름 | 표의 모든 행이 점으로 찍힙니다 |

<sub>표 3-5. 단계마다 더한 것과 달라진 것</sub>

### ⑦ 관제권 그리기

이제 점이 아니라 '범위'를 그려 봅시다. 공항 주변의 관제권은 비행 승인이 필요한 대표적인 구역입니다(항공안전법 제127조). 관제권은 보통 공항 중심에서 반지름 9.3 km입니다.

지도 위에 원을 그리려면 '9.3 km'를 위도와 경도로 바꿔야 합니다. 위도 1도의 길이는 어디서나 약 111 km입니다. 경도 1도의 길이는 적도에서 약 111 km이지만 극지방으로 갈수록 짧아집니다. 111에 cos(위도)를 곱하면 그 위도에서의 길이가 나옵니다. 이 두 값으로 공항에서 9.3 km 떨어진 점을 5도 간격으로 찍고, 점들을 선으로 이으면 원이 됩니다. 점을 찍는 px.scatter\_map() 대신 선을 긋는 px.line\_map()을 쓰고, 공항 자리의 점은 fig.add\_scattermap()으로 그 위에 더합니다.

**예제 3-12. 공항 관제권 지도 그리기**

In [ ]:
# 예제 3-12. 공항 관제권 지도 그리기
import math
import plotly.express as px

airport = (36.7166, 127.4991)      # 청주국제공항(위도, 경도), 지도 서비스에서 확인한 대략의 값
radius = 9.3                       # 관제권 반지름(km)

km_lat = 111                                           # 위도 1도의 길이(km)
# 이 위도에서 경도 1도의 길이(km): 111에 cos(위도)를 곱한다
km_lon =

# 공항에서 9.3 km 떨어진 점을 5도 간격으로 찍는다
lats, lons = [], []
for angle in range(0, 361, 5):
    rad = math.radians(angle)
    lats.append(airport[0] + radius / km_lat * math.cos(rad))
    lons.append(airport[1] + radius / km_lon * math.sin(rad))

fig = px.line_map(lat=lats, lon=lons, zoom=10, height=500,
                  center={"lat": airport[0], "lon": airport[1]})    # 점들을 이은 선: 실제 거리로 그린 원
fig.update_traces(line_color="red", name="관제권(반지름 9.3 km)", showlegend=True)
fig.add_scattermap(lat=[airport[0]], lon=[airport[1]], mode="markers",
                   marker_size=20, marker_color="blue",            # 비교용: 화면 픽셀 단위
                   name="청주국제공항")
fig.update_layout(map_style="carto-positron")                      # 배경 지도 고르기
fig.show()

print("경도 1도의 길이:", round(km_lon, 1), "km")

실행 결과(지도 아래에 나오는 줄)

```
경도 1도의 길이: 89.0 km
```

코랩에서는 fig.show()가 셀 아래에 지도를 띄웁니다. 마우스 휠로 지도를 확대·축소해 보세요. 빨간 원은 지도를 키우면 함께 커지고, 파란 점은 언제나 같은 크기로 보입니다. 청주에서는 경도 1도가 89 km이므로, 같은 9.3 km라도 경도로는 위도보다 더 큰 각도만큼 움직여야 합니다.

> **짚고 가기 · 선으로 그린 원과 점은 크기의 단위가 다릅니다.**  
> 빨간 원은 실제 땅 위의 거리(km)를 위도·경도로 바꿔 그렸으므로 지도를 확대하거나 축소해도 9.3 km 범위를 지킵니다. 파란 점의 marker\_size는 화면의 점 크기(픽셀)입니다. 관제권처럼 실제 거리를 나타낼 때 점의 크기를 키워서 그리면 확대할 때마다 엉뚱한 범위를 보여 줍니다. 같은 '원'이라도 단위를 확인하지 않으면 어떤 일이 생기는지 보여 주는 예입니다.

> **짚고 가기 · 지도가 비어 있다면**  
> 배경 지도는 실행할 때마다 인터넷에서 받아 옵니다. 지도가 나오지 않거나 바탕이 비어 있으면 인터넷 연결을 확인하고 셀을 다시 실행합니다.

### ====== 해 보기 ======

**우리 학교 주변 지도 만들기**

1. 지도 서비스에서 학교와 주변의 주요 지점 네 곳을 고르고 위도·경도를 적습니다. (이용 조건을 먼저 확인합니다.)
2. ④처럼 네 지점을 딕셔너리에 담고, ⑤를 고쳐 학교를 중심으로 한 지도에 네 지점을 점으로 표시합니다.
3. 학교에서 가장 가까운 공항을 찾아 관제권을 원으로 그려 봅니다. 학교가 그 안에 들어가는지 확인합니다.
4. 학교 주변에 실제 비행 제한 구역이 있는지 공식 자료에서 확인하고, 지도에 그린 결과와 비교합니다.
5. 공공데이터포털(data.go.kr)에서 위도와 경도가 열로 들어 있는 CSV 파일을 하나 골라 pd.read\_csv()로 읽고, ⑥처럼 지도에 올려 봅니다. 열 이름이 '위도', '경도'가 아닐 수 있으므로 df.columns로 먼저 확인합니다. 읽을 때 UnicodeDecodeError가 나면 pd.read\_csv(파일 이름, encoding="cp949")로 읽습니다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 공간정보는 좌표, 지형, 건물, 제한 구역 등 '어디에 무엇이 있는가'에 대한 정보입니다.
- 좌표를 알면 math.dist()로 거리를, 구간의 합으로 총 거리를 구할 수 있습니다.
- 제한 구역은 중심과 반지름으로 단순화해 포함 여부를 판단할 수 있습니다.
- 그림과 지도로 나타내면 숫자만으로는 보이지 않던 경로의 문제를 찾을 수 있습니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 3-4 배송 경로와 거점 후보 검토

**학습 목표**

- 여러 기준을 세워 후보를 비교·검토할 수 있습니다.
- 기준의 중요도를 정해 종합 점수를 계산할 수 있습니다.
- 선택의 근거를 설명하고 다른 의견과 비교해 토의할 수 있습니다.

<sub>관련 성취기준 [드코1 03-04-01] / [드코1 03-04-02]</sub>

> **생각 열기**  
> 배송 거점을 세 곳 가운데 한 곳에 지으려 합니다. A는 비행 제한 구역에서 가장 멀어 안전하고, B는 배송지까지 고르게 가깝고, C는 평균 거리가 가장 짧고 바람도 적습니다. 어디를 골라야 할까요? 그리고 왜 그렇게 골랐는지 어떻게 설명할 수 있을까요?

<sub>교재에서 읽기: ■ 1. 정답이 하나가 아닌 문제 · ■ 2. 평가 기준 세우기</sub>

## ■ 3. 후보 비교하기

**예제 3-13. 거점 후보의 기준별 값 구하기**

In [ ]:
# 예제 3-13. 거점 후보의 기준별 값 구하기
import math
import pandas as pd

stops = [(3.6, 4.2), (6.6, 4.0), (7.8, 1.8)]          # 배송지 좌표(km)
center, radius = (5.0, 6.0), 1.25                     # 제한 구역
candidates = {"A": (1.2, 2.0), "B": (5.5, 2.5), "C": (7.0, 3.0)}
wind = {"A": 3.1, "B": 4.6, "C": 2.4}                 # 연습용 평균 풍속(m/s)

rows = []
for name, xy in candidates.items():
    # 후보 지점에서 각 배송지까지의 거리 리스트
    dists =
    rows.append({
        "후보": name,
        "평균거리": ,        # 거리의 합 ÷ 개수 (소수 둘째 자리)
        "최대거리": ,        # 가장 먼 배송지까지의 거리 (소수 둘째 자리)
        "제한여유": round(math.dist(xy, center) - radius, 2),
        "평균풍속": wind[name],
    })
df = pd.DataFrame(rows)
print(df)

실행 결과

```
  후보  평균거리  최대거리  제한여유  평균풍속
0  A  5.21  6.60  4.27   3.1
1  B  2.27  2.55  2.29   4.6
2  C  2.04  3.61  2.36   2.4
```

표를 보면 어느 후보도 모든 기준에서 가장 좋지는 않습니다. C는 평균 거리가 가장 짧고 바람도 적지만, 가장 먼 배송지까지는 B보다 멉니다. A는 제한 구역에서 멀지만 배송지까지 거리가 멉니다. 이처럼 기준끼리 서로 맞서는 상황을 어떻게 다룰지가 핵심입니다. 대괄호 안의 [math.dist(xy, s) for s in stops]는 3-3의 예제 3-11과 같이 '배송지마다 거리를 구해 리스트로' 만드는 문장입니다.

## ■ 4. 중요도를 정해 종합하기

기준마다 중요도(가중치)를 정해 하나의 점수로 모으는 방법이 있습니다. km와 m/s처럼 단위가 다른 값을 바로 더할 수는 없으므로, 먼저 0에서 1 사이로 고쳐 줍니다. 이를 정규화라고 합니다. 가장 좋은 후보가 1, 가장 나쁜 후보가 0이 되게 합니다.

```python
\text{점수} = \frac{\text{값} - \text{최솟값}}{\text{최댓값} - \text{최솟값}}
```

작을수록 좋은 기준(거리, 바람)은 1에서 이 점수를 빼서 뒤집습니다.

**예제 3-14. 중요도를 반영한 종합 점수**

In [ ]:
# 예제 3-14. 중요도를 반영한 종합 점수
import math
import pandas as pd

stops = [(3.6, 4.2), (6.6, 4.0), (7.8, 1.8)]
center, radius = (5.0, 6.0), 1.25
candidates = {"A": (1.2, 2.0), "B": (5.5, 2.5), "C": (7.0, 3.0)}
wind = {"A": 3.1, "B": 4.6, "C": 2.4}

rows = []
for name, xy in candidates.items():
    dists = [math.dist(xy, s) for s in stops]
    rows.append({"후보": name,
                 "평균거리": round(sum(dists) / len(dists), 2),
                 "제한여유": round(math.dist(xy, center) - radius, 2),
                 "평균풍속": wind[name]})
df = pd.DataFrame(rows)

# 0~1 점수로 바꾸기(정규화). 1이 가장 좋은 값이 되도록 한다.
d = df["평균거리"]
df["거리점수"] =        # 작을수록 좋음
g = df["제한여유"]
df["여유점수"] =        # 클수록 좋음
w = df["평균풍속"]
df["바람점수"] =        # 작을수록 좋음

weight = {"거리": 0.5, "여유": 0.3, "바람": 0.2}             # 합이 1이 되게 정한다
df["종합점수"] = (df["거리점수"] * weight["거리"]
                + df["여유점수"] * weight["여유"]
                + df["바람점수"] * weight["바람"]).round(3)

print(df[["후보", "거리점수", "여유점수", "바람점수", "종합점수"]].round(3))
print("\n선택:", df.sort_values("종합점수", ascending=False).iloc[0]["후보"], "후보")

실행 결과

```
  후보   거리점수   여유점수   바람점수   종합점수
0  A  0.000  1.000  0.682  0.436
1  B  0.927  0.000  0.000  0.464
2  C  1.000  0.035  1.000  0.711

선택: C 후보
```

같은 모양의 정규화 식을 세 번 썼습니다. 'AI 드론 지능형 제어'에서 def를 배우면 이 식에 이름을 붙여 한 번만 쓸 수 있습니다.

> **짚고 가기 · 중요도는 누가 정하는가**  
> 위에서 거리에 0.5, 제한 여유에 0.3, 바람에 0.2를 주었습니다. 이 숫자는 데이터가 알려 준 것이 아니라 사람이 정한 것입니다. 안전을 가장 중요하게 본다면 제한 여유에 0.6을 줄 수도 있고, 그러면 결과가 바뀔 수 있습니다.  
> 그러므로 결과를 발표할 때는 반드시 중요도를 어떻게 정했고 왜 그렇게 정했는지 함께 밝힙니다. 이것을 숨기면 '데이터가 그렇게 말했다'며 사람의 판단을 데이터 뒤에 감추는 셈이 됩니다.

## ■ 5. 중요도를 바꿔 보기

중요도를 바꾸면 결과가 얼마나 달라지는지 확인해 보는 것은 매우 중요합니다. 조금만 바꿔도 순위가 뒤집힌다면, 그 결론은 그만큼 불안정하다는 뜻입니다.

**예제 3-15. 중요도에 따라 결과가 어떻게 달라지는가**

In [ ]:
# 예제 3-15. 중요도에 따라 결과가 어떻게 달라지는가
import pandas as pd

# 예제 3-14에서 구한 점수를 그대로 쓴다
score = pd.DataFrame({
    "후보": ["A", "B", "C"],
    "거리점수": [0.000, 0.927, 1.000],
    "여유점수": [1.000, 0.000, 0.035],
    "바람점수": [0.682, 0.000, 1.000],
})
views = {
    "효율 중시": {"거리": 0.7, "여유": 0.1, "바람": 0.2},
    "안전 중시": {"거리": 0.2, "여유": 0.6, "바람": 0.2},
    "균형": {"거리": 0.34, "여유": 0.33, "바람": 0.33},
}
for view, w in views.items():
    total = (score["거리점수"] * w["거리"]
             + score["여유점수"] * w["여유"]
             + score["바람점수"] * w["바람"])
    # 종합 점수가 가장 큰 후보: idxmax()
    best =
    values = ", ".join(f"{n}={v:.2f}" for n, v in zip(score["후보"], total))
    print(f"{view:8s} → {values} ⇒ 선택: {best}")

실행 결과

```
효율 중시    → A=0.24, B=0.65, C=0.90 ⇒ 선택: C
안전 중시    → A=0.74, B=0.19, C=0.42 ⇒ 선택: A
균형       → A=0.56, B=0.32, C=0.68 ⇒ 선택: C
```

idxmax()는 가장 큰 값이 있는 줄의 번호를 알려 줍니다. zip()은 후보 이름과 점수를 한 쌍씩 묶어 주고, ", ".join(…)은 그 쌍들을 쉼표로 이어 한 줄로 만듭니다. :.2f는 '소수 둘째 자리까지'라는 표시입니다.

관점에 따라 답이 달라졌습니다. 안전을 중시하면 A가, 효율을 중시하거나 균형을 잡으면 C가 선택됩니다. 어느 것도 틀린 답이 아닙니다. 중요한 것은 자신이 어떤 관점을 택했는지 밝히고, 그 관점을 택한 까닭을 설명하는 것입니다.

> **짚고 가기 · B는 왜 한 번도 뽑히지 않았을까요?**  
> 표를 다시 봅시다. B는 거리(0.927 &lt; 1.000), 제한 여유(0.000 &lt; 0.035), 바람(0.000 &lt; 1.000) 세 기준 모두에서 C보다 나바습니다. 이런 후보는 중요도를 어떻게 바꿔도 1등이 될 수 없습니다. 모든 기준에서 다른 후보에게 뒤처지는 후보는 비교에서 일찍 걸러낼 수 있습니다. 후보가 많을 때 이런 후보를 먼저 지우면 검토가 훨씬 간단해집니다. 다만 예제 3-13에서 본 '최대거리'를 기준에 넣으면 B가 가장 좋으므로, 기준을 무엇으로 고르느냐에 따라 이 결론도 달라집니다.

## ■ 6. 근거를 갖춘 발표하기

검토 결과를 발표할 때는 다음 형식을 따르면 설득력이 높아집니다.

| 단계 | 내용 | 예 |
|---|---|---|
| 결론 | 무엇을 골랐는가 | 거점 C를 추천합니다 |
| 기준 | 무엇을 보고 골랐는가 | 평균 거리, 제한 구역 여유, 평균 풍속 세 가지를 보았습니다 |
| 근거 | 숫자로 설명 | C는 평균 거리 2.04 km로 가장 짧고 평균 풍속도 2.4 m/s로 가장 낮습니다 |
| 한계 | 무엇이 아쉬운가 | 다만 제한 구역 여유가 A의 절반 수준이라 경로 설계에 주의가 필요합니다 |
| 대안 | 다른 선택의 가능성 | 안전을 더 중시한다면 A가 알맞습니다 |

<sub>표 3-7. 근거를 갖춘 발표의 구성</sub>

### ====== 해 보기 ======

**거점 선정 토의하기**

1. 모둠별로 중요도를 정하고, 그렇게 정한 까닭을 한 문장으로 적습니다.
2. 예제 3-14를 고쳐 모둠의 중요도로 종합 점수를 계산합니다.
3. 표 3-7의 형식에 맞춰 발표 자료를 만듭니다.
4. 다른 모둠과 결과를 비교합니다. 결과가 다르다면 왜 달랐는지 토의합니다.
5. 토의한 뒤 처음 정한 중요도를 바꾸고 싶어졌다면 그 까닭을 적어 봅시다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 정답이 하나가 아닌 문제에서는 답보다 답을 고른 근거가 중요합니다.
- 평가 기준은 숫자로 나타낼 수 있고, 후보마다 달라야 하며, 목적과 관련되어야 합니다.
- 단위가 다른 기준은 정규화한 뒤 중요도를 곱해 종합 점수를 구합니다.
- 중요도는 사람이 정하는 것이므로 반드시 함께 밝히고, 바꿔 보며 결론이 흔들리지 않는지 확인합니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# Ⅲ단원 마무리

### 배운 내용 한눈에 보기

| 소단원 | 꼭 기억할 것 |
|---|---|
| 3-1 드론 비행 시뮬레이션 | 시나리오를 먼저 설계하고, 조건은 한 번에 하나만 바꾸어 반복 실행합니다. 결과는 시뮬레이션 규칙의 범위 안에서만 의미가 있습니다. |
| 3-2 비행 결과 비교와 시각화 | 목적에 맞는 그래프를 고르고 제목·축 이름과 단위·범례를 갖춥니다. 관찰 → 해석 → 근거와 한계의 순서로 설명합니다. |
| 3-3 지도와 공간정보 활용 | 좌표로 거리와 경로의 길이를 구하고, 제한 구역을 지도 위에 그려 숫자만으로는 보이지 않던 문제를 찾습니다. |
| 3-4 배송 경로와 거점 후보 검토 | 기준을 정규화하고 중요도를 곱해 후보를 비교합니다. 중요도를 함께 밝히고, 바꿔 보며 결론이 흔들리지 않는지 확인합니다. |

### 스스로 점검하기

단원을 시작할 때 세운 목표를 얼마나 이루었는지 표시해 봅니다. ○는 '혼자 할 수 있다', △는 '책을 보면 할 수 있다', ✕는 '아직 어렵다'입니다.

| 점검 내용 | ○ △ ✕ |
|---|---|
| 비행 시나리오를 설계하고 시뮬레이션으로 구현할 수 있다. |  |
| 조건을 바꿔 반복 실행하고 비행 기록을 모을 수 있다. |  |
| 비행 결과를 그래프로 나타내 비교하고 차이의 원인을 설명할 수 있다. |  |
| 지도와 공간정보를 활용해 경로와 거점을 검토하고 근거를 제시할 수 있다. |  |

△나 ✕에 표시한 항목은 관련 소단원의 핵심 정리와 예제로 돌아가 다시 살펴봅니다.